In [102]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [103]:
url = "https://www.joehallock.com/edu/COM498/surveydata.html"
html = requests.get(url)

In [104]:
soup = BeautifulSoup(html.text, 'html.parser')

In [105]:
tables = soup.find_all('table')
len(tables)

19

In [123]:
rows = []
for tr in tables[7].find_all("tr"):
    cols = []
    for td in tr.find_all('td'):
        text = td.get_text(strip = True)
        if text != '':
            cols.append(text.strip())
    if cols:
        rows.append(cols)

In [124]:
df = pd.DataFrame(rows)
df = df[4:236]
df = df.iloc[:, 0:28]
df.columns = ['Country', 'Age', 'Gender', 'Cheap', 'Reliable', 'Trust', 'Security', 'Speed', 'Fun',
              'Quality', 'Technology', 'Loneliness', 'Fear', 'Disease', 'Courage', 'Favorite',
              'Least Favorite', 'Shopping Freq', 'Communicate Freq', 'Research Freq', 'Selling Freq',
              'News Freq', 'Weather Freq', 'Games Freq', 'Music Freq', 'Movie Freq', 'Art Freq', 'Word Freq']
df['Country'] = df['Country'].str.replace("U.S.A.", "United States")
df

,Country,Age,Gender,Cheap,Reliable,Trust,Security,Speed,Fun,Quality,...,Communicate Freq,Research Freq,Selling Freq,News Freq,Weather Freq,Games Freq,Music Freq,Movie Freq,Art Freq,Word Freq
4,United States,26,Male,Brown,White,White,Blue,Grey,Yellow,Black,...,5,4,1,3,2,1,5,4,5,2
5,Philippines,22,Female,Brown,White,Yellow,Blue,Red,Yellow,Black,...,5,5,1,3,3,1,3,3,1,1
6,United States,25,Female,Orange,Blue,Green,Grey,Red,Purple,Black,...,4,5,1,5,3,4,4,1,2,1
7,United States,35,Male,Yellow,Black,Grey,Grey,Red,Orange,Black,...,5,5,1,5,4,3,5,1,5,5
8,United States,27,Female,Grey,Black,Green,Blue,Red,Orange,Black,...,4,4,1,4,2,2,3,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
231,United States,27,Male,Brown,Blue,Green,White,Red,Yellow,Grey,...,3,3,2,5,5,5,5,3,2,1
232,United States,24,Female,Brown,Black,Black,White,Black,Orange,Grey,...,4,2,2,2,3,4,4,4,5,2
233,United States,21,Male,Brown,Black,White,White,Blue,Orange,Purple,...,5,1,2,4,2,3,5,3,1,3
234,Germany,25,Female,White,Orange,Green,Green,Red,Yellow,Purple,...,4,3,1,4,4,3,4,3,2,2


In [108]:
def is_light_dark(color):
    if color == 'White':
        return 'Light'
    elif color == 'Yellow':
        return 'Light'
    elif color == 'Blue':
        return 'Dark'
    elif color == 'Purple':
        return 'Dark'
    elif color == 'Brown':
        return 'Dark'
    elif color == 'Red':
        return 'Mid'
    elif color == 'Green':
        return 'Mid'
    elif color == 'Orange':
        return 'Mid'
    elif color == 'Black':
        return 'Dark'
    elif color == 'Grey':
        return 'Mid'
    else:
        return color

df['LightDark'] = df['Favorite'].apply(is_light_dark)

In [109]:
url = "https://en.wikipedia.org/wiki/List_of_sovereign_states_by_male_to_female_income_ratio"
header = {'User-Agent':"Data Aquisition Project/0.1 (flutelys@byu.edu)"}
html = requests.get(url, headers = header)
soup = BeautifulSoup(html.text, 'html.parser')

In [122]:
table = soup.find('table')
rows = []
for tr in table.find_all('tr'):
    columns = []
    for td in tr.find_all('td'):
        if td.get_text(strip = True) != '':
            columns.append(td.get_text(strip = True))
    rows.append(columns)
income = pd.DataFrame(rows)[1:]
income.columns = ['Country', 'Male', 'Female', 'Ratio']
income['Country'] = income['Country'].str.extract(r"(\w*\s?\-?\w*\s?\w*\s?\w*)")
income['Male'] = income['Male'].str.replace(",", "")
income['Female'] = income['Female'].str.replace(",", "")
long = income.melt(id_vars = ['Country'], value_vars = ['Male', 'Female'],
            var_name = 'Gender', value_name = 'Income')
long

,Country,Gender,Income
0,Yemen,Male,1877
1,Iraq,Male,22332
2,Iran,Male,27375
3,Syria,Male,6688
4,Jordan,Male,15296
...,...,...,...
363,Thailand,Female,18717
364,Gambia,Female,2561
365,Botswana,Female,15531
366,Bahamas,Female,28999


In [127]:
df['Country'] = df['Country'].str.strip()
long['Country'] = long['Country'].str.strip()
df['Gender'] = df['Gender'].str.strip()
long['Gender'] = long['Gender'].str.strip()
pd.merge(df, long, on = ['Country', 'Gender'])#, how = 'left')

,Country,Age,Gender,Cheap,Reliable,Trust,Security,Speed,Fun,Quality,...,Research Freq,Selling Freq,News Freq,Weather Freq,Games Freq,Music Freq,Movie Freq,Art Freq,Word Freq,Income
0,United States,26,Male,Brown,White,White,Blue,Grey,Yellow,Black,...,4,1,3,2,1,5,4,5,2,87081
1,Philippines,22,Female,Brown,White,Yellow,Blue,Red,Yellow,Black,...,5,1,3,3,1,3,3,1,1,7744
2,United States,25,Female,Orange,Blue,Green,Grey,Red,Purple,Black,...,5,1,5,3,4,4,1,2,1,60085
3,United States,35,Male,Yellow,Black,Grey,Grey,Red,Orange,Black,...,5,1,5,4,3,5,1,5,5,87081
4,United States,27,Female,Grey,Black,Green,Blue,Red,Orange,Black,...,4,1,4,2,2,3,1,1,1,60085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,United States,27,Male,Brown,Blue,Green,White,Red,Yellow,Grey,...,3,2,5,5,5,5,3,2,1,87081
224,United States,24,Female,Brown,Black,Black,White,Black,Orange,Grey,...,2,2,2,3,4,4,4,5,2,60085
225,United States,21,Male,Brown,Black,White,White,Blue,Orange,Purple,...,1,2,4,2,3,5,3,1,3,87081
226,Germany,25,Female,White,Orange,Green,Green,Red,Yellow,Purple,...,3,1,4,4,3,4,3,2,2,52189
